In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('100_Unique_QA_Dataset.csv')

In [3]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [4]:
# tokenize
def tokenize(text):
    text = text.lower()
    text = text.replace('?', '')
    text = text.replace('"', '')
    return text.split()

In [5]:
tokenize("What is the capital of France?")

['what', 'is', 'the', 'capital', 'of', 'france']

In [6]:
# vocab
vocab = {'<UNK>': 0}
def build_vocab(row):
    tokenize_question = tokenize(row['question'])
    tokenize_answer = tokenize(row['answer'])

    merged_token = tokenize_question + tokenize_answer

    for token in merged_token:
        if token not in vocab:
            vocab[token] = len(vocab)

In [7]:
df.apply(build_vocab, axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [8]:
len(vocab)

326

In [9]:
# covert words to numerical indices
def text_to_indices(text, vocab):
    indexed_text = []

    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    return indexed_text

In [10]:
text_to_indices("What is campusx", vocab)

[1, 2, 0]

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader

In [12]:
class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
        
    def __getitem__(self, index):
        numeric_question =  text_to_indices(self.df.iloc[index]['question'], self.vocab)
        numeric_answer =  text_to_indices(self.df.iloc[index]['answer'], self.vocab)

        return torch.tensor(numeric_question), torch.tensor(numeric_answer)
    def __len__(self):
        return self.df.shape[0]

In [13]:
dataset = QADataset(df, vocab)

In [14]:
dataset[0]

(tensor([1, 2, 3, 4, 5, 6]), tensor([7]))

In [15]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [16]:
for question, answer in dataloader:
    print(question, answer)

tensor([[ 10,  75, 209]]) tensor([[210]])
tensor([[ 1,  2,  3, 50, 51, 19,  3, 45]]) tensor([[52]])
tensor([[10, 96,  3, 97]]) tensor([[98]])
tensor([[ 42, 301, 302, 118,  14, 303, 304, 159, 305, 306, 307, 308]]) tensor([[309]])
tensor([[ 42, 137,   2, 227, 143,   3, 228, 229]]) tensor([[156]])
tensor([[  1,   2,   3, 147, 148,  19, 149]]) tensor([[150]])
tensor([[ 10,  29, 130, 131]]) tensor([[132]])
tensor([[  1,   2,   3,   4,   5, 238, 239]]) tensor([[240]])
tensor([[  1,   2,   3, 147,  86,  19, 193, 194]]) tensor([[195]])
tensor([[ 10, 310,   3, 311, 312]]) tensor([[313]])
tensor([[ 42, 217, 118, 218, 219,  19,  14, 220,  43]]) tensor([[221]])
tensor([[1, 2, 3, 4, 5, 8]]) tensor([[9]])
tensor([[ 10,  75, 111]]) tensor([[112]])
tensor([[ 42, 201,   2,  14, 202, 203, 204, 205]]) tensor([[206]])
tensor([[  1,   2,   3,   4,   5, 113]]) tensor([[114]])
tensor([[  1,   2,   3,   4,   5, 288]]) tensor([[289]])
tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([[91]])
tensor([[  1,   2,   3

In [17]:
import torch.nn as nn

In [27]:
class simpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, question):
        embedded_question =  self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))

        return output

In [32]:
lr = 0.001
epochs = 100

In [33]:
model = simpleRNN(len(vocab))

In [34]:
criterion = nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=lr)

In [35]:
# training loops
for epoch in range(epochs):
    total_loss = 0

    for question, answer in dataloader:
        optim.zero_grad()

        # forward pass
        output = model(question)

        # loss -> output shape(1, 324) - (1)
        loss = criterion(output, answer[0])

        # gradients
        loss.backward()

        optim.step()

        # update
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss}")


Epoch 1, Loss: 524.3816499710083
Epoch 2, Loss: 460.5488328933716
Epoch 3, Loss: 383.1447653770447
Epoch 4, Loss: 317.62412667274475
Epoch 5, Loss: 264.2619981765747
Epoch 6, Loss: 214.71507620811462
Epoch 7, Loss: 168.45653545856476
Epoch 8, Loss: 129.28130328655243
Epoch 9, Loss: 97.93346592783928
Epoch 10, Loss: 73.84117993712425
Epoch 11, Loss: 55.54392510652542
Epoch 12, Loss: 43.40524795651436
Epoch 13, Loss: 34.17384420335293
Epoch 14, Loss: 27.65657190978527
Epoch 15, Loss: 22.570784009993076
Epoch 16, Loss: 18.849912650883198
Epoch 17, Loss: 15.954392835497856
Epoch 18, Loss: 13.476064763963223
Epoch 19, Loss: 11.669276729226112
Epoch 20, Loss: 10.123310461640358
Epoch 21, Loss: 8.95868144184351
Epoch 22, Loss: 7.903004962950945
Epoch 23, Loss: 6.972823314368725
Epoch 24, Loss: 6.225234586745501
Epoch 25, Loss: 5.598997436463833
Epoch 26, Loss: 5.0550566129386425
Epoch 27, Loss: 4.594341095536947
Epoch 28, Loss: 4.193475259467959
Epoch 29, Loss: 3.822903836145997
Epoch 30, Los

In [47]:
def predict(model, question, threshold=0.5):
    # convert question to numbers
    num_question = text_to_indices(question, vocab)

    # tensor
    question_tensor = torch.tensor(num_question).unsqueeze(0)

    # send to model
    output = model(question_tensor)

    # convert logits to prob
    prob = nn.functional.softmax(output, dim=1)

    # find max prob
    value, index = torch.max(prob, dim=1)

    if value < threshold:
        print("I don't know.")
        
    list(vocab.keys())[index]

In [46]:
predict(model, "What is the capital of paris")

I don't know.


In [ ]:
# dataset[0][0]

tensor([1, 2, 3, 4, 5, 6])

In [ ]:
# x = nn.Embedding(324, embedding_dim=50)

In [ ]:
# a = x(dataset[0][0])

In [ ]:
# y = nn.RNN(50, 64)

In [ ]:
# y(a)[0].shape # hidden state

torch.Size([6, 64])

In [ ]:
# b = y(a)[1] # final output

In [ ]:
# z = nn.Linear(64, 324)

In [ ]:
# z(b)

tensor([[-0.2185,  0.3970,  0.2737, -0.0626, -0.1251, -0.1159,  0.0659,  0.1891,
         -0.3969, -0.3392,  0.4673,  0.1156,  0.1829,  0.0844,  0.1548, -0.1652,
         -0.2916,  0.1107,  0.0073,  0.0357,  0.1857, -0.1549,  0.0345, -0.1128,
         -0.0300,  0.0747,  0.0951, -0.3375, -0.3442,  0.0331, -0.3015,  0.0950,
          0.2718, -0.1483, -0.2085, -0.0902, -0.0919,  0.0013,  0.0659, -0.1852,
          0.0974,  0.0577, -0.1320, -0.0077,  0.0557, -0.0134,  0.2586,  0.1508,
         -0.1941, -0.2442,  0.2913,  0.5474, -0.3058, -0.1563, -0.4606,  0.0112,
          0.3216,  0.2673,  0.3609, -0.3616, -0.1502,  0.0570,  0.1113,  0.5685,
          0.0243,  0.2505,  0.1407,  0.1850, -0.5316,  0.0694, -0.0427,  0.1072,
          0.1338,  0.2524, -0.0780, -0.0873,  0.3051, -0.3385,  0.4252,  0.4798,
         -0.1806,  0.3579, -0.2321, -0.1723, -0.1178,  0.0978, -0.0779, -0.2079,
         -0.0334, -0.0624,  0.0984, -0.1003, -0.2506,  0.1864,  0.0496, -0.1708,
          0.1952, -0.0692,  